# Welcome to our city housing exploration

This is a hands-on tour of one question — *is our city building the homes it promised?* — answered not from a press release but from the city's own published data, one checkable step at a time.

It's written for a **curious beginner: you don't need to know how to code to start.** If you can open a notebook and run it top to bottom, you can follow every step here — and by the end you'll do something most people never get to: take a real number you computed yourself and set it beside the city's own report to the state, to see whether they agree.

## How to run this

There are two ways to run these notebooks, and **either one works** — the notebook adapts to wherever it finds itself:

- **In the cloud (easiest):** open it in **Google Colab** — nothing to install, nothing to download. Click and go.
- **On your own machine:** run it in **Jupyter** locally, if you'd rather keep your own copy and tinker.

Either way, the notebook **pulls in Berkeley's open data automatically** as its worked examples — you never have to hunt down or download a spreadsheet yourself. The first code cell sets that up; just run it.

*Coming soon — more cities.* Oakland and San Francisco publish their data through an open **API** (a standing web service you can query directly), so they're easy to add — we'll bring them in soon. Berkeley currently doesn't offer one, which is why we work from its permit **spreadsheet exports** instead. Same destination, slightly different on-ramp.

### Running the cells

To run a cell, click it and press **Shift + Return**, or click the **run (▸) button** on the cell. The simplest way through any notebook here is to start at the top and run each cell in order, reading the output that appears beneath it.

Some of the computational cells may look complex right now — that's expected, and it's fine. **You don't need to understand every line yet;** the ideas become clear as you go. Run them, watch what they produce, and keep moving.

💡 Tip: the **Next** link opens the following notebook in a new tab. If Colab says you have too many sessions, just close the previous tab and continue.

<!-- NAV:auto-generated by scripts/build_nav.py — do not edit by hand -->

← [Back to the course](https://berkeleybuild.com/data-science-curriculum.html)  |  Next: [JN0b · What a computational notebook is](https://colab.research.google.com/github/blockXblock/berkeley-housing-analysis/blob/main/notebooks/curriculum/JN0b_notebook.ipynb) →

# JN0a · Why municipal data

*On-ramp 1 of 8 — read before JN1.*

Berkeley made a promise to the State of California: over eight years, make room for about **9,000 new homes**. Did it happen? Who actually checks — and how would *you*, personally, find out?

That question is why this course exists. The answer is buried in data the city already publishes — but *published* and *checkable* are not the same thing. This notebook is about why that gap matters, and why, for the first time, someone who has never written a line of code can close it.

## (run first) Colab setup

Fetches the data + shared modules from R2. **No-op if you already have the repo locally.** On Colab it recreates the minimal layout so the cells below find everything.

In [1]:
# === COLAB BOOTSTRAP - fetch curriculum data + modules from R2 (NO-OP if the repo is local) ===
from pathlib import Path
import sys, urllib.request, urllib.parse, tarfile, subprocess

R2 = 'https://pub-2cee87f70da64080ab70ee0a34b55099.r2.dev/curriculum'
USE_CLEAN = False   # False: raw .xlsx path (JN1's messy-data lesson).  True (skip-ingest): permits_clean.*

_here = Path.cwd()
_have_repo = (_here/'scripts'/'build_v2').exists() or any((p/'scripts'/'build_v2').exists() for p in _here.parents)

def _get(url):
    # r2.dev sits behind Cloudflare, which 403s the default 'Python-urllib' User-Agent; send a browser UA.
    req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
    with urllib.request.urlopen(req, timeout=60) as r:
        return r.read()

if _have_repo:
    print('local repo detected - no fetch needed')
else:
    try:
        import pyarrow  # the parquet / USE_CLEAN path needs it; Colab has pandas, maybe not pyarrow
    except ImportError:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'pyarrow'], check=True)
    def _fetch(url, dest):
        dest = Path(dest)
        if dest.exists():
            return                                   # cached: re-runs don't re-download
        dest.parent.mkdir(parents=True, exist_ok=True)
        dest.write_bytes(_get(url)); print('fetched', dest.name)
    # 1) shared modules -> ./scripts/...  (the config-cell repo-root walk then finds scripts/build_v2)
    if not (_here/'scripts'/'build_v2').exists():
        Path('modules.tgz').write_bytes(_get(f'{R2}/curriculum_modules.tar.gz'))
        _tar = tarfile.open('modules.tgz')
        try: _tar.extractall(_here, filter='data')      # py3.12+: safe extract, no deprecation warning
        except TypeError: _tar.extractall(_here)         # older python has no filter arg
        _tar.close(); Path('modules.tgz').unlink(missing_ok=True)   # tidy: drop the intermediate tarball
        print('extracted modules -> ./scripts/')
    # 2) data -> the SAME relative paths the notebooks use (raw .xlsx AND clean exports, both fetched)
    for rel in ['data/raw/cpra-downloads/BP_Annual Permit Report-2018-2022.xlsx',
                'data/raw/cpra-downloads/BP_Annual Permit Report-2023-2025.xlsx',
                'databases/hcd_apr_mirror_2026-06-17_fresh.db',
                'databases/hcd_apr_mirror.db',
                'data/processed/permits_clean.csv',
                'data/processed/permits_clean.parquet',
                'data/processed/permits_clean_README.md']:
        _fetch(f"{R2}/data/{urllib.parse.quote(rel.split('/')[-1])}", _here/rel)   # quote -> %20 for the spaced .xlsx names
    print('curriculum bundle ready (fetched from R2)')


local repo detected - no fetch needed


In [2]:
def md(t):
    from IPython.display import Markdown, display
    display(Markdown(t))

## What's a repo?

The code below "walks up" the folders to find the **repo root** — so before it runs, it's worth knowing what a repo is.

A **repository** — "repo" for short — is the project's folder of files, but with a superpower: it's kept under *version control*, a powerful tool that records **every change and update** over the life of the project. Nothing is quietly lost — you can see what changed, when, and why, compare any two moments, and roll back if something goes wrong. That tracked history is what lets a project like this one stay trustworthy. This whole course lives in one repo.

A couple of terms you'll meet right away:

- **Repo root** — the single top folder that everything else sits inside. The notebooks find the data by their *position relative to* this root, so they keep working no matter where you put the project. (That's exactly what the next code cell does.)
- **Clone** — your own local copy of the repo, made when you want to run things on your own machine, tinker, or keep a personal copy. You **don't** need to clone to start: Colab can run a notebook for you without one. Clone when you want to settle in; skip it when you just want to look.

*First-time setup, gently.* If you do keep a local copy, put the project somewhere tidy and easy to find — a dedicated folder you'll remember, not buried among random downloads. A clean root matters because everything here is located *relative to* that top folder: keep it intact and the notebooks will always find their data. We stay deliberately light on commands here — later, **JN0g** and **JN0h** show how an AI agent actually works *inside* a repo like this one.

## Point the notebook at the data

Finds the repo root, locates the permit feed, and puts the project's real shared code on the path. The two knobs near the top are all a student changes to run this on another city.

In [3]:
# === CONFIG — point this at YOUR city's permit data (this notebook is clonable) ===
from pathlib import Path
import sys, glob

# walk up to the repo root (where scripts/build_v2 lives) so the notebook runs from anywhere
REPO_ROOT = Path.cwd().resolve()
while not (REPO_ROOT / 'scripts' / 'build_v2').exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent

# --- the two knobs a student changes for another city ---
PERMIT_GLOB   = str(REPO_ROOT / 'data/raw/cpra-downloads/BP_Annual Permit Report-*.xlsx')
HEADER_ROW    = 7        # 0-indexed: Berkeley's CPRA export puts the column names on row 8
EXPECTED_UNIQUE = 30764  # the known unique-permit total for YOUR feed (Berkeley = 30,764)

# import the REAL shared modules the pipeline uses (we demonstrate them, never reinvent)
sys.path.insert(0, str(REPO_ROOT / 'scripts'))
sys.path.insert(0, str(REPO_ROOT / 'scripts' / 'build_v2'))
print('repo root :', REPO_ROOT)
print('feed files:', [Path(f).name for f in glob.glob(PERMIT_GLOB)])


repo root : /Users/johngage/berkeley-data
feed files: ['BP_Annual Permit Report-2023-2025.xlsx', 'BP_Annual Permit Report-2018-2022.xlsx']


## What is "open data," and why should you care?

**Open data** is information a government publishes for anyone to use — often because the law (a *public records* or *CPRA* request, in California) requires it. A **building permit** is one such record: the city's permission slip to build, renovate, or demolish. Every new home in Berkeley starts as one of these rows.

So the raw material is *right there*. Let's open it and see what "right there" actually looks like.

In [4]:
import pandas as pd, glob
_f = sorted(glob.glob(PERMIT_GLOB))[0]            # take the first permit spreadsheet in the feed
raw = pd.read_excel(_f, header=None, dtype=str)   # read it with NO assumed header — exactly as the city shipped it
print('the raw file is', raw.shape[0], 'rows x', raw.shape[1], 'columns — here is the very top:')
raw.head(8)                                        # peek at the top rows (title + blanks sit above the real columns)

the raw file is 18061 rows x 26 columns — here is the very top:


,0,1,2,3,4,5,6,7,8,9,...,16,17,18,19,20,21,22,23,24,25
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,BP Annual Permit Report,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,For Post Date: 1/1/2018 to 12/31/2022,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,PermitNumber,Submittal Date,Issuance Status,NaN,Issuance Date,Finaled Status,Finaled Date,Completed,Completed Date,Parcel Number,...,NaN,ADU,Detached,Work Type,OccType,SubType,NumberUnits,UnitsAdded,UnitsRemoved,CO Required


In [5]:
# find which of the top rows actually holds the column names
_hdr = next(i for i in range(9) if 'PermitNumber' in raw.iloc[i].astype(str).tolist())
md(f'''### A pile of data is not an answer

Read with no assumptions, the top of the file is mostly junk — a title line and blank rows. The real column names (`PermitNumber`, `StreetNumber`, …) don't even appear until **row {_hdr}** (counting from 0). The city *published* this — but published is not the same as **legible**: *can an ordinary person get a defensible answer out of it?* Not yet. Making it legible is **JN1**'s job; here we're just looking.''')

### A pile of data is not an answer

Read with no assumptions, the top of the file is mostly junk — a title line and blank rows. The real column names (`PermitNumber`, `StreetNumber`, …) don't even appear until **row 7** (counting from 0). The city *published* this — but published is not the same as **legible**: *can an ordinary person get a defensible answer out of it?* Not yet. Making it legible is **JN1**'s job; here we're just looking.

In [6]:
def _load(p):
    # read one spreadsheet at its real header row, then tidy the column names
    d = pd.read_excel(p, dtype=str, header=HEADER_ROW); d.columns = [str(c).strip() for c in d.columns]; return d
df = pd.concat([_load(f) for f in glob.glob(PERMIT_GLOB)], ignore_index=True)   # stack every yearly file into one table
df = df[df['PermitNumber'].notna()].copy()        # drop rows that have no permit number
# count the rows, the distinct permits, and how many are tagged New construction
n_rows = len(df); n_unique = df['PermitNumber'].nunique(); n_new = int((df['Work Type'] == 'New').sum())
n_other = n_unique - n_new                          # everything else: alterations, additions, demolitions, signs
md(f'''### Two numbers that already complicate the question

Loaded as a real table, the feed holds **{n_rows:,}** rows — **{n_unique:,}** distinct permits across 2018–2025. But only **{n_new:,}** are tagged **New** construction. The other ~**{n_other:,}** are alterations, additions, demolitions, even signs — real city activity, but not new homes. So even *how many permits?* immediately becomes *how many of which kind?* — and learning to ask that precisely is the whole course.''')

### Two numbers that already complicate the question

Loaded as a real table, the feed holds **32,202** rows — **30,764** distinct permits across 2018–2025. But only **1,773** are tagged **New** construction. The other ~**28,991** are alterations, additions, demolitions, even signs — real city activity, but not new homes. So even *how many permits?* immediately becomes *how many of which kind?* — and learning to ask that precisely is the whole course.

## The arc this series walks

From this messy feed we'll build, step by step: raw permits → a clean table → **buildings** (many permits describe one building) → **completed homes** → and finally a number we can set beside *the city's own report to the state* and check. Each notebook is one honest step, and every step shows its work.

## Why *you* can do this now

Two things changed. **Notebooks** (next) let prose and live code sit together, so every number carries its proof. And **AI agents** (JN0g–JN0h) let you direct that code in plain English. You don't need to become a programmer — you need to learn to ask precise questions and check the answers. That's a skill, not a degree.

**Next — JN0b:** the notebook itself — what it is, and why re-running it *is* re-checking it.

<!-- NAV:auto-generated by scripts/build_nav.py — do not edit by hand -->

← [Back to the course](https://berkeleybuild.com/data-science-curriculum.html)  |  Next: [JN0b · What a computational notebook is](https://colab.research.google.com/github/blockXblock/berkeley-housing-analysis/blob/main/notebooks/curriculum/JN0b_notebook.ipynb) →